下列是 orderV2 的常用 API (按顺序执行)

在开始前, 我们需要知道一些缩略词和定义:
ISU: Integrated Sign Up
PPCP: PayPal Commerce Platform
这2个都是针对平台类型的卖家, 让非平台类型的卖家, 授权给平台类型的卖家代为call API的

OrderV2: 是指的创建订单的那一组API, 其中的url中会有一个 `/v2/`

In [ ]:
import requests
# import path
import sys
import uuid
import json
import base64
import ipykernel
import pandas
import re

# ensure that ipykernel version 6.0.0 or greater
print("kernel version:",ipykernel.__version__)

# Ensure that you're using a kernel based on Python 3.7 or greater.
print("Python version:",sys.version)

kernal version: 6.17.1
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
# 使用你们自己创建的clientID 和 secret
client_id = r'AYqf2JlRXxl1qB2zxFReBIKc4QEVYbBdCiMzuOfCZEwqxIENKT9ecmR-dwQWHuyDmMCrDicNmZJyaVxK'
secret_key = r'EINpXkA4L0O-4sm3D2xxhdUsftVBEJkdQQNmyrkGVx7RVGVySKwh89F882LYO8o7diknzk949r3wdi-h'

##### Default Test Info ######
# Merchant: c2-ipynb-test@merchant.com; AccountID: 326N4C8A6CKSY; Password: 12345678; Create-Acct: email_computer@163.com
# Platform: c2-ipynb-test@platform.com; AccountID: C5759NYTKH7ZJ; Password: 12345678; Create-Acct: email_computer@163.com; bncode: C2-ipynb-code(setup manually)


# API部分
**对于平台类型的商户来说,在获取了Platform权限后,需要先引导商户完成商户入驻**

## 商户入驻

### 1.Get Access Token
获取一个9小时过期的token, 你也可以使用basic auth的加密方式, 即参数中使用 `auth=(client_id, secret_key)`  
但是在前端的请求中这会暴露自己的用户名密码给到买家用户, 存在安全隐患

In [3]:
params = {'grant_type': "client_credentials"}
url = r'https://api.sandbox.paypal.com/v1/oauth2/token'
headers = {
    "Content-Type": r"application/x-www-form-urlencoded"
}


response_result = requests.post(
    url=url, auth=(client_id, secret_key), params=params, headers=headers)

# print(json.dumps(response_result.json(), indent=2, ensure_ascii=False))
access_token = response_result.json()["access_token"]
print("access_token:",access_token)


access_token: A21AALbsK0YsIcWp_RwdppptHIu7l8WlwK69sNCOLU1yPHc_kqiCHnSjTddDwJhJoOSX9dNF8OwGVQoK6CFuvH_VcCp6uu6xw


In [4]:
headers = {
    "Content-Type": r"application/json",
    'Authorization': 'Bearer ' + access_token
}

In [5]:
def print_response(response_result, is_print_raw_text=False):
    if is_print_raw_text:
        print(response_result.text)
    print(response_result.status_code)
    print(json.dumps(response_result.json(), indent=2, ensure_ascii=False))


### 2.Partner Referrals
这一步是用于获取授权非平台类型的卖家的授权链接

In [6]:
url = r"https://api.sandbox.paypal.com/v2/customer/partner-referrals"


onboard_tracking_id = uuid.uuid4().__str__()
print("onboard_tracking_id:",onboard_tracking_id)

request_data = {
    "tracking_id": onboard_tracking_id,
    "operations": [
        {
            "operation": "API_INTEGRATION",
            "api_integration_preference": {
                "rest_api_integration": {
                    "integration_method": "PAYPAL",
                    "integration_type": "THIRD_PARTY",
                    "third_party_details": {
                        "features": [
                            "PAYMENT",
                            "REFUND",
                            "ADVANCED_TRANSACTIONS_SEARCH",
                            "ACCESS_MERCHANT_INFORMATION",
                            "TRACKING_SHIPMENT_READWRITE"
                        ]
                    }
                }
            }
        }
    ],
    "products": [
        # 个人卖家无法使用PPCP, 请参考这里
        # https://developer.paypal.com/docs/api/partner-referrals/v2/#partner-referrals_create!path=products&t=request
        "EXPRESS_CHECKOUT"
        # "PPCP"
    ],
    "legal_consents": [
        {
            "type": "SHARE_DATA_CONSENT",
            "granted": True
        }
    ],
    "partner_config_override": {
        "partner_logo_url": r"https://www.paypalobjects.com/webstatic/mktg/logo/pp_cc_mark_111x69.jpg",

        #需要注意, url需要符合RFC-1035的规范. 具体可以参考 https://www.freesoft.org/CIE/RFC/1035/6.htm
        "return_url": r"https://www.bing.com"
    }
}


response_result = requests.post(
    url=url,   headers=headers, json=request_data)

print_response(response_result)
links = response_result.json()["links"]
# print(links)
action_url = ''
for link_obj in links:
    if link_obj["rel"] == 'action_url':
        action_url = link_obj["href"]
print("action_url:",action_url)


onboard_tracking_id: be9665c5-37ab-46b3-9fb7-d0b92a82af67
201
{
  "links": [
    {
      "href": "https://api.sandbox.paypal.com/v2/customer/partner-referrals/NWYyOWY2ZDUtNjhhZi00OTlkLTk0ZWYtYzE0ZDM0OTBkMzA4RSsyVlJWajREMVltaUFNWWo4YXlaa3lVSXkxYXd0T0JoeE1oZjZ6b2VSZz12Mg==",
      "rel": "self",
      "method": "GET",
      "description": "Read Referral Data shared by the Caller."
    },
    {
      "href": "https://www.sandbox.paypal.com/bizsignup/partner/entry?referralToken=NWYyOWY2ZDUtNjhhZi00OTlkLTk0ZWYtYzE0ZDM0OTBkMzA4RSsyVlJWajREMVltaUFNWWo4YXlaa3lVSXkxYXd0T0JoeE1oZjZ6b2VSZz12Mg==",
      "rel": "action_url",
      "method": "GET",
      "description": "Target WEB REDIRECT URL for the next action. Customer should be redirected to this URL in the browser."
    }
  ]
}
action_url: https://www.sandbox.paypal.com/bizsignup/partner/entry?referralToken=NWYyOWY2ZDUtNjhhZi00OTlkLTk0ZWYtYzE0ZDM0OTBkMzA4RSsyVlJWajREMVltaUFNWWo4YXlaa3lVSXkxYXd0T0JoeE1oZjZ6b2VSZz12Mg==


### 3.让商家使用该action_url来完成onboard

如果没有测试卖家账户的话,可以使用这个信息
```yml
email address: petro-test01-us-bs@cctest.com
pwd: Qq111222333
```

**** 2026年更新 ****  
使用这个测试商户: c2-ipynb-test@merchant.com | 12345678


### 4.获取merchantID (也可以在第三步中通过return_url跳转回来的时候在url params中获取)

In [7]:
# HardCode
partner_id = r'C5759NYTKH7ZJ' # c2-ipynb-test@platform.com
url = f'https://api.sandbox.paypal.com/v1/customer/partners/{partner_id}/merchant-integrations?tracking_id={onboard_tracking_id}'

response_result = requests.get(url=url,   headers=headers)
print_response(response_result)

onboard_merchant_id = response_result.json()['merchant_id']
print("onboard_merchant_id:", onboard_merchant_id)


404
{
  "name": "RESOURCE_NOT_FOUND_ERROR",
  "debug_id": "6202234f8ee1e"
}


KeyError: 'merchant_id'

### 5.Get Seller onboarding status (在这里可以获取显示用的primary_email,payments_receivable等参数)

In [ ]:


url = f'https://api.sandbox.paypal.com/v1/customer/partners/{partner_id}/merchant-integrations/{onboard_merchant_id}'

response_result = requests.get(url=url,   headers=headers)
print_response(response_result)


在上面的request的response中, 需要注意, 如果是在给商户进行高级信用卡支付(ACDC)的权限申请的时候, 关注`capabilities`这个字段下, `GUEST_CHECKOUT`,`APPLE_PAY`,`APPLE_PAY`的状态是否为`ACTIVE`. 因为有的商户可能没有ACDC的权限

Onboard到这里就结束了, 下面是创建订单和收款等流程

## 订单创建和收款  

  
如果你是一个direct merchant, 也就是说没有通过在平台中进行商户入驻的方式(第一部分的流程). 那么请忽略请求头中的 `PayPal-Auth-Assertion`,`PayPal-Partner-Attribution-Id`

### 6.Create order 创建一个订单

In [ ]:
addressObj = {
    "name": {
        "given_name": "PayPal",
        "surname": "Test Customer",
    },
    "address": {
        "address_line_1": "123 ABC Street",
        "address_line_2": "Apt 2",
        "admin_area_2": "San Jose",
        "admin_area_1": "CA",
        "postal_code": "95121",
        "country_code": "US",
    },
    "email_address": "petro-test01-us@cctest.com",
    "password": "Qq111222333",
    "phone": {
        "phone_type": "MOBILE",
        "phone_number": {
            "national_number": "14082508100",
        },
    },
}

print(type(addressObj["name"]))

In [ ]:
shippingAddrObj = {
    "shipping": {
        "address": {
            "address_line_1": "2211 N First Street",
            "address_line_2": "Building 17",
            "admin_area_2": "San Jose",
            "admin_area_1": "CA",
            "postal_code": "95131",
            "country_code": "US"
        }
    }
}


**请注意,如果您作为平台主体,在代为授权完成的商家操作的时候, 订单/交易部分的API都需要加上 PayPal_Auth_Assertion. 这可以理解成是用于告知服务器我现在正在操作的商户是哪一个**

In [ ]:

def create_paypal_auth_assertion():
    # 我这里选择了一个已经入驻好的商家id作为默认值(如果我没有执行前面的cell), 如果已经完成了前面的步骤, 那这里的变量不会为空
    if "onboard_merchant_id" not in locals().keys():
        onboard_merchant_id = r"326N4C8A6CKSY" # c2-ipynb-test@merchant.com; AccountID: 326N4C8A6CKSY

    to_encode = {
        "iss": client_id,
        "payer_id": onboard_merchant_id
    }

    normal_string = json.dumps(to_encode)
    print((type(normal_string)))

    encoded_string = base64.b64encode(normal_string.encode("utf-8"))
    b64_string = encoded_string.decode("utf-8")
    # print(b64_string)
    assertion = f'eyJhbGciOiJub25lIn0=.{b64_string}.'
    print("PayPal_Auth_Assertion:",assertion)
    return  assertion



In [ ]:
PayPal_Auth_Assertion = create_paypal_auth_assertion()

headers = {
    "Content-Type": r"application/json",
    'Authorization': 'Bearer ' + access_token,
    # 请使用自己的BNCODE, 这里hard code了
    "PayPal-Partner-Attribution-Id": "C2-ipynb-code",

    # request-id是一个实现幂等性的参数, 在网络抖动的情况下, 同样的request-id不会重复创建订单
    "PayPal-Request-Id": "0001",
    "PayPal-Auth-Assertion":PayPal_Auth_Assertion
}

### >>请仔细阅读这个部分的参数注解来明白语义, 通常情况下, 这些参数是最少的参数量. 传递少于这些信息量的订单会导致风控

In [ ]:

isAddressSet = True
request_data = {
    "intent": "CAPTURE",
    "purchase_units": [
        {

            "reference_id": "000010000200003",
            # 商品描述, 请尽量使用标准英文字符
            "description" : "This is a test product"
            "amount": {
                "currency_code": "USD",
                "value": "100.00",
            },
        },
    ],
    "payment_source": {

        # 如果你使用了信用卡支付或者其他的支付方式, 请传递对应的值. https://developer.paypal.com/docs/api/orders/v2/#orders_create!ct=application/json&path=payment_source&t=request
        "paypal": {

            # 不同的payment_source下的字段是不一样的, paypal中比较多, 卡支付的相对较少
            "experience_context": {
                # 有UNRESTRICTED和IMMEDIATE_PAYMENT_REQUIRED 2个值, 如果不想接受类似echeck这样的延迟支付, 可以选择IMMEDIATE_PAYMENT_REQUIRED
                "payment_method_preference": "IMMEDIATE_PAYMENT_REQUIRED",

                # 在buyer的支付页面中显示的店铺名字
                "brand_name": "YOUR[EXAMPLE INC]BRAND Name",
                "locale": "en-US",


                "landing_page": "LOGIN",
                "shipping_preference": "SET_PROVIDED_ADDRESS" if isAddressSet else "GET_FROM_FILE",

                # 有CONTINUE和PAY_NOW 2个值, 如果在快捷支付的情况下, 也就是类似在商品详情页就支付, 之后再算运费再capture的情况下, 可以用CONTINUE
                "user_action": "PAY_NOW",

                # return_url和风控模型息息相关
                "return_url": "https://example.com/returnUrl",
                "cancel_url": "https://example.com/cancelUrl",
            },
        },
    },
}


if isAddressSet:
    # 在paypal.experience_context下的地址是账单地址
    request_data["payment_source"]["paypal"]["experience_context"].update(addressObj)
    # 在purchase_units下的地址是物流地址
    request_data["purchase_units"][0].update(shippingAddrObj)

print(json.dumps(request_data, indent=2, ensure_ascii=False))


In [ ]:
url = r'https://api.sandbox.paypal.com/v2/checkout/orders'
response_result = requests.post(url=url,   headers=headers, json=request_data)

print(json.dumps(response_result.json(), indent=2, ensure_ascii=False))

payer_approve_url = ''
for link_obj in response_result.json()['links']:
    if link_obj["rel"] == 'payer-action':
        action_url = link_obj["href"]
print("action_url:",action_url)

# 把这个orderId以字符串形式传给前端的onApprove函数
order_id =  response_result.json()['id']
print("order_id:",order_id)

根据具体业务情况的不同, 可以选择先让buyer approve, 也可以在查看订单详情做过其他验证后再让buyer approve. 在这里, create_order的response中有`links`字段, 下面的对象就是当前这个订单可以执行的操作. 在有些情况下, 一个订单不能被`approve` (也就是没有`payer-action`这个`href`

### 7.Show order details 查看订单详情

In [ ]:
url = f'https://api.sandbox.paypal.com/v2/checkout/orders/{order_id}'
print(url)

if "PayPal-Request-Id" in headers:
    headers.pop("PayPal-Request-Id")

headers

In [ ]:
response_result = requests.get(url=url,   headers=headers)

print_response(response_result)

### 8.Update order (更新订单, 比如在下单后因为需要调整运费的时候使用/这个功能在部分地区比如欧洲可能会收到相关法规限制(PSD2之类))

In [ ]:
url = f'https://api.sandbox.paypal.com/v2/checkout/orders/{order_id}'

item = {
    "op": "replace",
    "path": "/purchase_units/@reference_id=='000010000200003'/amount",
            "value": {
                "value": "115",
                "currency_code": "USD"
            }
}

request_data = [item]

response_result = requests.patch(url=url,   headers=headers, json=request_data)

# 这个接口没有返回值, 如果更新正确就是204
print(response_result.status_code)

### 9.Capture Order (完成订单 )

In [ ]:
url = f'https://api.sandbox.paypal.com/v2/checkout/orders/{order_id}/capture'

# 这个接口如果成功, 则资金发生变动
# 注意, BNCODE(PayPal-Partner-Attribution-Id)这个字段在Capture中是必须, 因为只有Capture这一步才是完成交易的步骤.
response_result = requests.post(url=url,   headers=headers)

In [ ]:
print_response(response_result)


In [ ]:
res_obj = json.loads(response_result.text)
# print(res_obj)


capture_id = res_obj["purchase_units"][0]["payments"]["captures"][0]["id"]
print(capture_id)


### 10.Get Capture Details (查看交易详情)

In [ ]:
url = f'https://api.sandbox.paypal.com/v2/payments/captures/{capture_id}'

response_result = requests.get(url=url,   headers=headers)
print_response(response_result=response_result)

注意, 这里的`capture_id`就是卖家的交易号, 和merchant在网页端中看到的是一致. `order_id`通常可以被理解为是订单号

# 订单/交易到这里就结束了, 下面是tracking和dispute部分

## Tracking API
上传物流信息在发生客诉的时候至关重要, 这是merchant发货的直接依据, 同时, 上传好的物流信息卖家买家都可以通过登录导PayPal中查看到.

### [Deprecated]老的Tracking API

#### [Tracking-API]01.Get Tracking Info

In [ ]:
tracking_id="123456"
url = f"https://api.sandbox.paypal.com/v1/shipping/trackers/{capture_id}-{tracking_id}"
response_result = requests.get(url=url,   headers=headers)
print_response(response_result=response_result)

### 新的Tracking API

## Invoice部分
如果你想让买家有更多元化的支付方式. 可以尝试账单功能
> https://developer.paypal.com/docs/invoicing/

### [Invoice-API]01.Generate Invoice Number(创建invoice号，这一步可以不做)

In [ ]:
url = r'https://api.sandbox.paypal.com/v2/invoicing/generate-next-invoice-number'

headers = {
    "Content-Type": r"application/json",
    'Authorization': 'Bearer ' + access_token,
    "PayPal_Auth_Assertion":PayPal_Auth_Assertion
}

response_result = requests.post(url=url,   headers=headers)
print_response(response_result=response_result)

### [Invoice-API]02.List Invoice

In [ ]:
url = r'https://api.sandbox.paypal.com/v2/invoicing/invoices'

headers = {
    "Content-Type": r"application/json",
    'Authorization': 'Bearer ' + access_token,
    "PayPal_Auth_Assertion":PayPal_Auth_Assertion
}

response_result = requests.get(url=url,   headers=headers)
# print_response(response_result=response_result)


In [ ]:
invoice_detail_lists = response_result.json()["items"]
print(len(invoice_detail_lists))
flat_invoice_items=[]
for it_invoice_detail in invoice_detail_lists:
    flat_invoice_item = dict()
    flat_invoice_item["id"] = it_invoice_detail["id"]
    flat_invoice_item["status"] = it_invoice_detail["status"]
    flat_invoice_item["currency_code"] = it_invoice_detail["detail"]["currency_code"]
    flat_invoice_item["invoice_number"] = it_invoice_detail["detail"]["invoice_number"]
    flat_invoice_item["invoicer"] = it_invoice_detail.get("invoicer",dict()).get("email_address","NO-INVOICER")
    flat_invoice_item["recipient"] = it_invoice_detail["primary_recipients"][0]["billing_info"]["email_address"]
    flat_invoice_items.append(flat_invoice_item)

# print(flat_invoice_items)
dataFrame = pandas.DataFrame(flat_invoice_items)
dataFrame

### [Invoice-API]03.Create Invoice Draft

In [ ]:
invoicer_email_address = r'petro-test01-us-bs@cctest.com'
recipient_email_address= r"petro-test01-es2@cctest.com"
request_data = {
     "detail": {

        "reference": "deal-ref",
        "invoice_date": "2020-11-12",
        "currency_code": "USD",
        "note": "Thank you for your business.",
        "term": "No refunds after 30 days.",
        "memo": "This is a long contract",
        "payment_term": {
            "term_type": "NET_10",
            "due_date": "2020-11-22"
        }
    },
    "invoicer": {
        "name": {
            "given_name": "MyTest-Invoicer",
            "surname": "invoicer-Surname"
        },
        "address": {
            "address_line_1": "1234 First Street",
            "address_line_2": "337673 Hillside Court",
            "admin_area_2": "Anytown",
            "admin_area_1": "CA",
            "postal_code": "98765",
            "country_code": "US"
        },
        "email_address": invoicer_email_address,
        "phones": [
            {
                "country_code": "001",
                "national_number": "4085551234",
                "phone_type": "MOBILE"
            }
        ],
        "website": "www.test.com",
        "tax_id": "001002003",
        "logo_url": "https://example.com/logo.PNG",
        "additional_notes": "2-4"
    },
    "primary_recipients": [
        {
            "billing_info": {
                "name": {
                    "given_name": "Test-Recipient",
                    "surname": "Paypal"
                },
                "address": {
                    "address_line_1": "1100 Congress Ave",
                    "admin_area_2": "CAustin",
                    "admin_area_1": "TX",
                    "postal_code": "78701",
                    "country_code": "US"
                },
                "email_address": recipient_email_address,
                "phones": [
                    {
                        "country_code": "001",
                        "national_number": "16503858068",
                        "phone_type": "HOME"
                    }
                ],
                "additional_info_value": "add-info"
            }
        }
    ],
    "items": [
        {
            "name": "A basket of Apple",
            "description": "Very Taste Apple",
            "quantity": "1",
            "unit_amount": {
                "currency_code": "USD",
                "value": "66.52"
            },
            "tax": {
                "name": "Sales Tax",
                "percent": "7.25"
            },
            "discount": {
                "percent": "5"
            },
            "unit_of_measure": "QUANTITY"
        },

    ],
    "configuration": {
        "partial_payment": {
            "allow_partial_payment": True,
            "minimum_amount_due": {
                "currency_code": "USD",
                "value": "20.00"
            }
        },
        "allow_tip": True,
        "tax_calculated_after_discount": True,
        "tax_inclusive": False
    },
    "amount": {
        "breakdown": {
            "custom": {
                "label": "Packing Charges",
                "amount": {
                    "currency_code": "USD",
                    "value": "10.00"
                }
            },
            "shipping": {
                "amount": {
                    "currency_code": "USD",
                    "value": "10.00"
                },
                "tax": {
                    "name": "Sales Tax",
                    "percent": "7.25"
                }
            },
            "discount": {
                "invoice_discount": {
                    "percent": "5"
                }
            }
        }
    }
}

In [ ]:
url = r'https://api.sandbox.paypal.com/v2/invoicing/invoices'
response_result = requests.post(url=url,   headers=headers, json=request_data)
print_response(response_result=response_result)


In [ ]:
invoice_link = response_result.json()["href"]
pattern = re.compile(r'(\w{4})-(\w{4})-(\w{4})-(\w{4})-(\w{4})')
match_result = pattern.search(invoice_link)
# print(m.group())
invoice_id = match_result.group()

### [Invoice-API]04.Show invoice Details

In [ ]:

url = f'https://api.sandbox.paypal.com/v2/invoicing/invoices/{invoice_id}'
response_result = requests.get(url=url,   headers=headers)
# print_response(response_result=response_result)
invoice_detail = dict()
invoice_detail["id"] = response_result.json()["id"]
invoice_detail["status"] = response_result.json()["status"]
invoice_detail["txn_id"] = response_result.json()["payments"]["transactions"][0]["payment_id"]
invoice_detail

### [Invoice-API]05.Send Invoice (Notify by email)(Email won't be sent in Sandbox)

In [ ]:
url = f'https://api.sandbox.paypal.com/v2/invoicing/invoices/{invoice_id}/send'
request_data={
    "send_to_invoicer": True
}
response_result = requests.post(url=url,   headers=headers, json=request_data)
print_response(response_result=response_result)

In [ ]:
tracking_id="12345"
url = f"https://api.sandbox.paypal.com/v1/shipping/trackers/{invoice_detail['txn_id']}-{tracking_id}"
print(url)
response_result = requests.get(url=url,   headers=headers)
print_response(response_result=response_result)

# 按钮展示Best Practice

## JSv5

### 1.Default Behavior
在正常情况下, 我们默认以Button Set的形式展示PayPal支付按钮,  
我们可以看到`PayPal`(也就是PayPal wallet钱包支付),`PayLater`(PayPal的先买后付),`Debit or Credit Card`(标准的信用卡支付, 我们也会称之为BCDC, basic credit&debit card checkout)

In [ ]:
## 请注意, 这里的createOrder和onApprove回调函数中的写法仅作简易展示使用, 实际业务中请务必不要使用这个做法, 使用上方的Orders API来完成订单的创建和capture

from IPython.display import display, HTML, Javascript


html_code = """
<div id="paypal-button-container"></div>
<p id="result-message"></p>
"""

js_code = f"""
(function() {{
    // 动态创建 script 标签引入 SDK
    var script = document.createElement('script');
    script.src = "https://www.paypal.com/sdk/js?client-id={client_id}&currency=USD";

    script.onload = function() {{
        // SDK 加载完成，配置按钮
        paypal.Buttons({{
            // 创建订单
            createOrder: function(data, actions) {{
                return actions.order.create({{
                    purchase_units: [{{
                        amount: {{
                            value: '0.01' // 测试金额
                        }}
                    }}]
                }});
            }},
            // 支付成功回调
            onApprove: function(data, actions) {{
                return actions.order.capture().then(function(details) {{
                    // 在页面显示成功信息
                    document.getElementById('result-message').innerText =
                        '支付成功！交易者: ' + details.payer.name.given_name;

                    // 可选：你甚至可以通过 google.colab.kernel.invokeFunction
                    // 把支付结果传回给 Python 变量（如果需要进阶交互）
                }});
            }},
            onError: function(err) {{
                console.error(err);
                document.getElementById('result-message').innerText = '支付出错，请查看控制台';
            }}
        }}).render('#paypal-button-container');
    }};

    document.head.appendChild(script);
}})();
"""

# 4. 渲染
print("正在加载 PayPal 按钮...")
display(HTML(html_code))
display(Javascript(js_code))

### 2.BCDC standalone
除了在default display中的信用卡支付按钮,  
通过调研, 我们也发现, 额外拆分一个信用卡按钮的支付选项对于支付成功率是有助的. 也就是standalone BCDC.

In [ ]:
## 请注意, 这里的createOrder和onApprove回调函数中的写法仅作简易展示使用, 实际业务中请务必不要使用这个做法, 使用上方的Orders API来完成订单的创建和capture

from IPython.display import display, HTML, Javascript


html_code = """
<div id="paypal-button-container"></div>
<p id="result-message"></p>
"""

js_code = f"""
(function() {{
    // 动态创建 script 标签引入 SDK
    var script = document.createElement('script');
    script.src = "https://www.paypal.com/sdk/js?client-id={client_id}&currency=USD";

    script.onload = function() {{
        // SDK 加载完成，配置按钮
        paypal.Buttons({{
            // 创建订单
            createOrder: function(data, actions) {{
                return actions.order.create({{
                    purchase_units: [{{
                        amount: {{
                            value: '0.01' // 测试金额
                        }}
                    }}]
                }});
            }},
            // 支付成功回调
            onApprove: function(data, actions) {{
                return actions.order.capture().then(function(details) {{
                    // 在页面显示成功信息
                    document.getElementById('result-message').innerText =
                        '支付成功！交易者: ' + details.payer.name.given_name;

                    // 可选：你甚至可以通过 google.colab.kernel.invokeFunction
                    // 把支付结果传回给 Python 变量（如果需要进阶交互）
                }});
            }},
            onError: function(err) {{
                console.error(err);
                document.getElementById('result-message').innerText = '支付出错，请查看控制台';
            }}
        }}).render('#paypal-button-container');
    }};

    document.head.appendChild(script);
}})();
"""

print("正在加载 PayPal 按钮...")
display(HTML(html_code))
display(Javascript(js_code))

### 3.PayLater Standalone
很多buyer不会把PayPal和PayLater看成一个brand. 我们也建议拆分单独的PayLater支付, 因为这可以更好地显示PayLaterMessage, 这有助于直观地向买家展示分期的金额